# Predictive Anayltics: Support Vector Machines with Regression

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [51]:
from run_config import PATHS

In [52]:
#TODO: embedding ansehen -> vielleicht austauschen
#TODO: make more time efficient
#TODO: change to thundersvm -> installation umstädlich
#TODO: funktion einbauen zum 
#TODO: vielleicht PCA adden

In [53]:
GRID_SAMPLE = 50_000 # if validation set over 50k use only 50k rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "HEXAGON" # HEXAGON
SPATIAL_ENCODING = "latlong" # options: embedding, latlong
MODE = "full" # options: full, sample
TIME_UNIT = "4H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [54]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [55]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder

#import networkx as nx
#from libpysal.weights import Queen
#from node2vec import Node2Vec

## Preparations

In [56]:
INPUT = PATHS.train_test_dir

In [57]:
# Paths
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"

In [58]:
MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [59]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [60]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-03-05 04:00:00,3,4,4,0.866025,0.500000,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
1,2026-03-05 12:00:00,3,4,12,0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,32.25,16.125,11.00,21.25,Prcard
2,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
3,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
4,2025-02-15 04:00:00,2,6,4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,2026-02-16 20:00:00,2,1,20,0.500000,0.866025,0.000000,1.000000,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71390,2025-02-13 04:00:00,2,4,4,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71391,2025-02-13 04:00:00,2,4,4,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71392,2025-12-28 04:00:00,12,7,4,-0.500000,0.866025,-0.781831,0.623490,8.660254e-01,0.5,...,0.0,2.0,0.4,0.0,2.0,128.63,25.726,5.98,54.90,Mobile


In [61]:
#print("Currently working on a sample from all the data due to runtime issues")
#train_df = train_df.sample(n=10_000, random_state=42)

In [62]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-03-05 04:00:00,3,4,4,0.866025,0.500000,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.0,0.00,No trips
1,2026-03-05 12:00:00,3,4,12,0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,32.25,16.125,11.0,21.25,Prcard
2,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.0,0.00,No trips
3,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.0,0.00,No trips
4,2025-02-15 04:00:00,2,6,4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.0,0.00,No trips


In [63]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [64]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [65]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-03-05 04:00:00,3,4,4,0.866025,0.500000,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
1,2026-03-05 12:00:00,3,4,12,0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,32.25,16.125,11.00,21.25,Prcard
2,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
3,2025-08-02 00:00:00,8,6,0,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
4,2025-02-15 04:00:00,2,6,4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,2026-02-16 20:00:00,2,1,20,0.500000,0.866025,0.000000,1.000000,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71390,2025-02-13 04:00:00,2,4,4,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71391,2025-02-13 04:00:00,2,4,4,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
71392,2025-12-28 04:00:00,12,7,4,-0.500000,0.866025,-0.781831,0.623490,8.660254e-01,0.5,...,0.0,2.0,0.4,0.0,2.0,128.63,25.726,5.98,54.90,Mobile


In [66]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: hexa")
    for df in (train_df, val_df, test_df):
        df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

Encoding: latlong and Unit: hexa


In [67]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else: 
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Onehot

In [68]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
   # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)

### Spatial Encoding: Spatial Embedding

In [69]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "HEXAGON"):

    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])

    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Features per H3 cell (mean of POI features, train only)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Required by srai: maps each region to its features
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


Create y

In [70]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [71]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,0.866025,0.500000,0.433884,-0.900969,8.660254e-01,0.5,0,7.943925,0,0,...,0,0,0,1,1,0,0,0.029200,-0.744235,0.667279
1,0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,0,10.649357,19,7,...,0,0,1,0,1,0,0,0.030801,-0.744628,0.666769
2,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,0,18.241517,4,35,...,0,1,0,0,1,0,0,0.030930,-0.743484,0.668038
3,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,0,16.441243,174,19,...,0,1,0,0,1,0,0,0.030549,-0.743551,0.667981
4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,0,4.678589,1,1,...,0,0,1,0,0,1,0,0.027346,-0.742759,0.669000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,0.500000,0.866025,0.000000,1.000000,-8.660254e-01,0.5,1,19.143988,44,0,...,0,0,1,0,0,1,0,0.029813,-0.742780,0.668871
71390,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,0,1.645593,18,3,...,0,0,1,0,1,0,0,0.029275,-0.744894,0.666541
71391,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,0,19.143988,44,0,...,0,1,0,0,0,1,0,0.029813,-0.742780,0.668871
71392,-0.500000,0.866025,-0.781831,0.623490,8.660254e-01,0.5,0,16.029433,636,53,...,0,0,0,1,1,0,0,0.030828,-0.743727,0.667773


### Grid Search

In [72]:
model = SVR()

In [73]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,0.866025,0.500000,0.433884,-0.900969,8.660254e-01,0.5,0,7.943925,0,0,...,0,0,0,1,1,0,0,0.029200,-0.744235,0.667279
1,0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,0,10.649357,19,7,...,0,0,1,0,1,0,0,0.030801,-0.744628,0.666769
2,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,0,18.241517,4,35,...,0,1,0,0,1,0,0,0.030930,-0.743484,0.668038
3,-0.500000,-0.866025,-0.974928,-0.222521,0.000000e+00,1.0,0,16.441243,174,19,...,0,1,0,0,1,0,0,0.030549,-0.743551,0.667981
4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,0,4.678589,1,1,...,0,0,1,0,0,1,0,0.027346,-0.742759,0.669000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,0.500000,0.866025,0.000000,1.000000,-8.660254e-01,0.5,1,19.143988,44,0,...,0,0,1,0,0,1,0,0.029813,-0.742780,0.668871
71390,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,0,1.645593,18,3,...,0,0,1,0,1,0,0,0.029275,-0.744894,0.666541
71391,0.500000,0.866025,0.433884,-0.900969,8.660254e-01,0.5,0,19.143988,44,0,...,0,1,0,0,0,1,0,0.029813,-0.742780,0.668871
71392,-0.500000,0.866025,-0.781831,0.623490,8.660254e-01,0.5,0,16.029433,636,53,...,0,0,0,1,1,0,0,0.030828,-0.743727,0.667773


In [ ]:
# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVR(max_iter=15_000, tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=15_000, tol=1e-2))
])

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel
param_grid_linear = {
    "regressor__svm__C": [0.01, 0.1, 1], # excluded C=10, 100 due to convergance issues
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3, 0.5],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [0.01, 0.1, 1],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3, 0.5],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

param_grid_poly = {
    "regressor__svm__C": [0.01, 0.1, 1],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3, 0.5],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise",
        n_iter=10
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.7067839022448329 best params: {'regressor__svm__epsilon': 0.3, 'regressor__svm__C': 1}


In [ ]:
X_val_grid

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
3711,1.000000e+00,6.123234e-17,0.781831,0.623490,-8.660254e-01,0.5,0,18.058670,21,27,...,0,1,0,0,1,0,0,0.031210,-0.743660,0.667830
13373,8.660254e-01,5.000000e-01,-0.433884,-0.900969,-8.660254e-01,-0.5,0,13.788435,8,0,...,1,0,0,0,0,0,1,0.031787,-0.746466,0.664664
21255,5.000000e-01,8.660254e-01,0.974928,-0.222521,8.660254e-01,-0.5,0,3.426495,28,2,...,0,0,0,0,1,0,0,0.029657,-0.744827,0.666598
32753,0.000000e+00,1.000000e+00,0.000000,1.000000,8.660254e-01,0.5,1,10.694965,18,4,...,0,0,0,0,1,0,0,0.030875,-0.745286,0.666029
66961,1.224647e-16,-1.000000e+00,0.433884,-0.900969,0.000000e+00,1.0,0,13.978311,7,0,...,0,0,0,0,1,0,0,0.030467,-0.746250,0.664968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61327,5.000000e-01,8.660254e-01,0.974928,-0.222521,0.000000e+00,1.0,0,18.095826,15,0,...,0,0,0,0,0,0,1,0.031991,-0.745984,0.665195
51973,1.000000e+00,6.123234e-17,0.974928,-0.222521,1.224647e-16,-1.0,0,2.468898,13,2,...,0,1,0,0,1,0,0,0.028894,-0.744960,0.666483
7144,1.000000e+00,6.123234e-17,-0.781831,0.623490,1.224647e-16,-1.0,0,15.899802,220,15,...,1,0,0,0,1,0,0,0.029888,-0.743442,0.668132
29081,5.000000e-01,-8.660254e-01,0.000000,1.000000,-8.660254e-01,0.5,0,11.899551,37,11,...,0,0,0,0,1,0,0,0.031080,-0.744803,0.666560


In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__svm__epsilon': 0.05, 'regressor__svm__C': 10}
Best CV score: 0.24803264599839103


### Train Model

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

: 

: 

In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 6.06623197,  1.34785082,  0.92840365, ..., -3.8927612 ,
       -0.86106653, -1.00588888])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 16.578284127035385
MSE: 4093.5482175240877
RMSE: 63.98084258216742
R2 Score: 0.790151368685712


In [ ]:
# save model
if SPATIAL_UNIT == "HEXAGON":
    dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + H3_RES + "_" + TIME_UNIT + "_svr.joblib")
    dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + H3_RES + "_"  + TIME_UNIT + "_svr.joblib")
else: 
    dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
    dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']